# Visualize per-cell kNN Jaccard dissimilarity for different k values
Visualize cell-wise kNN Jaccard dissimilarity on Harmony UMAP
for two different k values (e.g. k=20 and k=500).

Author: Tengxiao Gao

Description:
This script visualizes precomputed per-cell dissimilarity values
(scVI vs Harmony) on the Harmony UMAP embedding.

In the previous script (single_pair_dissimilarity_k50_to_2000.py),
dissimilarity values were mapped onto the scVI UMAP.
Here, the purpose is to project the same dissimilarity values onto
the Harmony UMAP for comparison.

This allows us to examine how differences between embeddings
appear under different reference coordinate systems (scVI vs Harmony).

Main steps:
- Load AnnData object (for Harmony UMAP coordinates)
- Load per-cell dissimilarity CSV files
- Match cells via cell_index
- Plot UMAP colored by dissimilarity
- Use a shared color scale for fair comparison

Inputs:
- Zarr file containing embeddings and UMAP (`adata.obsm`)
- Per-cell CSV files with dissimilarity values

Outputs:
- UMAP plots for different k values (e.g. k=20, k=500)

Usage:
Simply update file paths at the top of the script and run.

Notes:
- Consistent color scale (vmin/vmax) is used across plots for comparability
- Assumes cell_index matches the row order of AnnData
- This script complements previous visualizations on scVI UMAP


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt


# =========================================================
# 1. paths
# =========================================================
RGC_PATH = "/vol/data/data/output/HRCA/RGC.zarr"

CSV_K20 = "RGC_all8_28pairs_scvi_lineage_vs_harmony_lineage_percell.csv"
CSV_K500 = "RGC_scvi_lineage_vs_harmony_lineage_k50_to_2000_percell_k500.csv"

OUT_K20 = "RGC_scvi_lineage_vs_harmony_lineage_on_harmonyUMAP_k20.png"
OUT_K500 = "RGC_scvi_lineage_vs_harmony_lineage_on_harmonyUMAP_k500.png"


# =========================================================
# 2. helper functions
# =========================================================
def find_obsm_key(adata, kind, name):
    """
    kind: 'emb' or 'umap'
    name: e.g. 'harmony_lineage'
    """
    candidates = [
        f"X_{kind}--metrics:{name}",
        f"X_{kind}--metrics/{name}",
        f"X_{kind}:{name}",
        f"X_{kind}/{name}",
        f"X_{kind}_{name}",
    ]
    for k in candidates:
        if k in adata.obsm_keys():
            return k

    # fallback: fuzzy search
    hits = [
        k for k in adata.obsm_keys()
        if (f"X_{kind}" in str(k)) and (name in str(k))
    ]
    if len(hits) == 1:
        return hits[0]
    elif len(hits) > 1:
        print(f"[find_obsm_key] Multiple candidates for {kind}, {name}: {hits}")
        return hits[0]

    return None


def get_umap_df_from_adata(adata, embedding_name):
    """
    Return a dataframe with:
    cell_index, UMAP1, UMAP2
    """
    umap_key = find_obsm_key(adata, "umap", embedding_name)
    if umap_key is None:
        raise ValueError(
            f"Could not find UMAP obsm key for '{embedding_name}'.\n"
            f"Available obsm keys:\n{list(adata.obsm_keys())}"
        )

    umap = adata.obsm[umap_key]
    if umap.shape[1] < 2:
        raise ValueError(f"UMAP in '{umap_key}' has shape {umap.shape}, expected at least 2 columns.")

    df_umap = pd.DataFrame({
        "cell_index": np.arange(adata.n_obs),
        "UMAP1": umap[:, 0],
        "UMAP2": umap[:, 1],
    })
    return df_umap, umap_key


def prepare_percell_df(csv_path, target_k):
    df = pd.read_csv(csv_path)

    needed = {"cell_index", "dissimilarity"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")

    if "k" in df.columns:
        df = df[df["k"] == target_k].copy()
        if len(df) == 0:
            raise ValueError(f"No rows with k={target_k} found in {csv_path}")

    return df


def merge_with_umap(df_percell, df_umap):
    df = df_percell.merge(df_umap, on="cell_index", how="left")

    n_missing = df["UMAP1"].isna().sum()
    if n_missing > 0:
        raise ValueError(
            f"{n_missing} rows could not be matched to UMAP coordinates by cell_index.\n"
            "This suggests cell_index does not align with the row order in the AnnData object."
        )
    return df


def plot_umap_colored(df, title, outpath, vmin=None, vmax=None):
    plt.figure(figsize=(9, 7), dpi=200)
    sc = plt.scatter(
        df["UMAP1"],
        df["UMAP2"],
        c=df["dissimilarity"],
        s=4,
        cmap="viridis",
        linewidths=0,
        vmin=vmin,
        vmax=vmax,
    )
    cbar = plt.colorbar(sc)
    cbar.set_label("cell-wise Jaccard dissimilarity")

    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, bbox_inches="tight")
    plt.show()


# =========================================================
# 3. load AnnData
# =========================================================
adata = ad.read_zarr(RGC_PATH)

print("adata shape:", adata.shape)
print("\nAvailable obsm keys:")
for k in adata.obsm_keys():
    print(" -", k)


# =========================================================
# 4. get harmony_lineage UMAP coordinates
# =========================================================
df_harmony_umap, harmony_umap_key = get_umap_df_from_adata(adata, "harmony_lineage")

print("\nUsing harmony UMAP key:", harmony_umap_key)
print(df_harmony_umap.head())


# =========================================================
# 5. read per-cell dissimilarity csv files
# =========================================================
df_k20 = prepare_percell_df(CSV_K20, target_k=20)
df_k500 = prepare_percell_df(CSV_K500, target_k=500)

print("\nk=20 df shape:", df_k20.shape)
print("k=500 df shape:", df_k500.shape)


# =========================================================
# 6. merge with harmony UMAP
# =========================================================
plot_df_k20 = merge_with_umap(df_k20, df_harmony_umap)
plot_df_k500 = merge_with_umap(df_k500, df_harmony_umap)

print("\nMerged k=20 shape:", plot_df_k20.shape)
print("Merged k=500 shape:", plot_df_k500.shape)


# =========================================================
# 7. optional: consistent color scale across both plots
# =========================================================
global_vmin = min(plot_df_k20["dissimilarity"].min(), plot_df_k500["dissimilarity"].min())
global_vmax = max(plot_df_k20["dissimilarity"].max(), plot_df_k500["dissimilarity"].max())

print("\nGlobal color scale:")
print("vmin =", global_vmin)
print("vmax =", global_vmax)


# =========================================================
# 8. plot k=20 on harmony UMAP
# =========================================================
plot_umap_colored(
    plot_df_k20,
    title="Dissimilarity on UMAP\nscvi_lineage vs harmony_lineage, k=20\nUMAP: harmony_lineage",
    outpath=OUT_K20,
    vmin=global_vmin,
    vmax=global_vmax,
)

# =========================================================
# 9. plot k=500 on harmony UMAP
# =========================================================
plot_umap_colored(
    plot_df_k500,
    title="Dissimilarity on UMAP\nscvi_lineage vs harmony_lineage, k=500\nUMAP: harmony_lineage",
    outpath=OUT_K500,
    vmin=global_vmin,
    vmax=global_vmax,
)

print("\nSaved:")
print(" -", OUT_K20)
print(" -", OUT_K500)